`Transformers and MLflow: Downstream Tasks and Experiment Tracking`

GitHub Repository: https://github.com/GeethikaSuharshani/transformers-mlflow-tutorial

`Task 1: Sentiment Classification`

In [1]:
import mlflow
import pandas as pd

from transformers import pipeline
from sklearn.metrics import accuracy_score

# Set up MLflow tracking
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Text Classification - Sentiment Analysis")

# Load pre-trained model for sentiment analysis
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [4]:
# Sample dataset of 20 sentences with their true labels for run 1
data_run1 = [
    ("I love this product, it's amazing!", "POSITIVE"),
    ("This is the worst experience ever.", "NEGATIVE"),
    ("Absolutely fantastic work!", "POSITIVE"),
    ("I hate this so much.", "NEGATIVE"),
    ("Very good and satisfying.", "POSITIVE"),
    ("Terrible service and bad quality.", "NEGATIVE"),
    ("I am extremely happy with this.", "POSITIVE"),
    ("This made me very disappointed.", "NEGATIVE"),
    ("What a great experience!", "POSITIVE"),
    ("I will never buy this again.", "NEGATIVE"),
    ("Highly recommended!", "POSITIVE"),
    ("Not worth the money.", "NEGATIVE"),
    ("I feel so good about this.", "POSITIVE"),
    ("It completely failed my expectations.", "NEGATIVE"),
    ("Brilliant and well executed.", "POSITIVE"),
    ("Worst purchase I have made.", "NEGATIVE"),
    ("I am impressed with the results.", "POSITIVE"),
    ("This is frustrating and useless.", "NEGATIVE"),
    ("Excellent quality and service.", "POSITIVE"),
    ("Very bad and disappointing.", "NEGATIVE"),
]

# Run inference and compute accuracy for run 1
texts = [t[0] for t in data_run1]
true_labels = [t[1] for t in data_run1]

preds = classifier(texts)
pred_labels = [p["label"] for p in preds]

accuracy = accuracy_score(true_labels, pred_labels)
print("Run 1 Accuracy:", accuracy)

# Log parameters, metrics, and artifacts to MLflow for run 1
with mlflow.start_run(run_name="run_1_baseline"):

    # Parameters
    mlflow.log_param("model_name", "distilbert-base-uncased-finetuned-sst-2-english")
    mlflow.log_param("num_samples", len(texts))
    mlflow.log_param("dataset_type", "clean_sentences")

    # Metrics
    mlflow.log_metric("accuracy", accuracy)

    # Create a DataFrame for predictions
    df = pd.DataFrame({
        "text": texts,
        "true_label": true_labels,
        "predicted_label": pred_labels
    })

    # Save predictions to CSV and log as artifact
    df.to_csv("run1_predictions.csv", index=False)
    mlflow.log_artifact("run1_predictions.csv")

Run 1 Accuracy: 1.0
🏃 View run run_1_baseline at: http://127.0.0.1:5000/#/experiments/6/runs/dff5269532b54105828357f12133043a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


In [5]:
# Sample dataset of 20 sentences with their true labels for run 2 (more complex sentences)
data_run2 = [
    ("Oh great, another perfect day ruined.", "NEGATIVE"),
    ("Wow, I just love waiting for hours.", "NEGATIVE"),
    ("Yeah right, this is totally amazing...", "NEGATIVE"),
    ("Best purchase ever... said no one.", "NEGATIVE"),
    ("I absolutely enjoyed wasting my money.", "NEGATIVE"),
    ("Fantastic, it broke after one day.", "NEGATIVE"),
    ("Sure, because this is exactly what I wanted.", "NEGATIVE"),
    ("I am so happy this failed again.", "NEGATIVE"),
    ("Totally brilliant, I can't even use it.", "NEGATIVE"),
    ("Oh wow, such incredible disappointment.", "NEGATIVE"),
    ("This is actually quite good.", "POSITIVE"),
    ("I really like how this works.", "POSITIVE"),
    ("Pretty decent overall experience.", "POSITIVE"),
    ("Not bad, I am satisfied.", "POSITIVE"),
    ("This exceeded my expectations.", "POSITIVE"),
    ("I am pleasantly surprised.", "POSITIVE"),
    ("Works better than I thought.", "POSITIVE"),
    ("Quite impressive results.", "POSITIVE"),
    ("I have no complaints.", "POSITIVE"),
    ("Really good quality product.", "POSITIVE"),
]

# Run inference and compute accuracy for run 2
texts = [t[0] for t in data_run2]
true_labels = [t[1] for t in data_run2]

preds = classifier(texts)
pred_labels = [p["label"] for p in preds]

accuracy = accuracy_score(true_labels, pred_labels)
print("Run 2 Accuracy:", accuracy)

# Log parameters, metrics, and artifacts to MLflow for run 2
with mlflow.start_run(run_name="run_2_sarcasm_harder_cases"):
    mlflow.log_param("model_name", "distilbert-base-uncased-finetuned-sst-2-english")
    mlflow.log_param("num_samples", len(texts))
    mlflow.log_param("dataset_type", "sarcasm_and_hard_cases")

    mlflow.log_metric("accuracy", accuracy)

    df = pd.DataFrame({
        "text": texts,
        "true_label": true_labels,
        "predicted_label": pred_labels
    })

    df.to_csv("run2_predictions.csv", index=False)
    mlflow.log_artifact("run2_predictions.csv")

Run 2 Accuracy: 0.65
🏃 View run run_2_sarcasm_harder_cases at: http://127.0.0.1:5000/#/experiments/6/runs/0447d149253c47918d32fd9ab9a4ee30
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


`Observations:`

Run 1 achieves perfect accuracy (1.0) on clean and straightforward sentiment sentences, indicating that the pretrained distilbert-base-uncased-finetuned-sst-2-english model performs very well on standard polarity classification tasks. The model effectively captures clear positive and negative sentiment when explicit lexical cues are present.

In contrast, Run 2 introduces sarcastic and contextually complex sentences, resulting in a significant drop in accuracy (0.65). This highlights the model's limitation in handling sarcasm and implicit sentiment, where positive words such as "great" or "amazing" are used in a negative context. The model relies heavily on surface-level lexical patterns rather than deeper contextual understanding, leading to misclassification in such cases.

Overall, the comparison demonstrates that while pretrained transformer models are highly effective on conventional sentiment data, their performance degrades when faced with nuanced linguistic phenomena such as sarcasm and contextual polarity shifts.

`Task 2: Named Entity Recognition`

In [2]:
import mlflow
import pandas as pd
from transformers import pipeline
from collections import Counter

# Set up MLflow tracking
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("NER - Named Entity Recognition")

# Initialize the NER pipeline with a pre-trained model
ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# Sample dataset of 10 paragraphs for NER
paragraphs_run1 = [
    "Elon Musk founded SpaceX in California. The company later expanded its operations to Texas.",
    "Microsoft is based in Seattle. It is one of the largest technology companies in the world.",
    "Apple is headquartered in Cupertino. It designs and sells the iPhone and MacBook globally.",
    "Barack Obama was the president of the United States. He served two terms in office.",
    "Google has offices in London and California. It specializes in search engines and AI systems.",
    "Amazon was started in Seattle by Jeff Bezos. It is now a global e-commerce giant.",
    "Tesla is led by Elon Musk. It focuses on electric vehicles and renewable energy solutions.",
    "The United Nations is located in New York. It works on global peace and cooperation.",
    "Facebook is now called Meta. It focuses on social media and virtual reality technologies.",
    "The World Bank operates globally. It provides financial support to developing countries."
]

entities_1 = []

with mlflow.start_run(run_name="ner_run_1_easy"):
    # Run NER on each paragraph and collect entities
    for p in paragraphs_run1:
        entities_1.extend(ner(p))

    labels = [e["entity_group"] for e in entities_1]
    counts = Counter(labels)

    mlflow.log_param("model_name", "dslim/bert-base-NER")
    mlflow.log_param("num_paragraphs", len(paragraphs_run1))
    mlflow.log_param("dataset_type", "easy_text")

    mlflow.log_metric("total_entities", len(entities_1))
    mlflow.log_metric("PER", counts.get("PER", 0))
    mlflow.log_metric("ORG", counts.get("ORG", 0))
    mlflow.log_metric("LOC", counts.get("LOC", 0))

    df = pd.DataFrame(entities_1)
    df.to_csv("ner_run1.csv", index=False)

    mlflow.log_artifact("ner_run1.csv")

🏃 View run ner_run_1_easy at: http://127.0.0.1:5000/#/experiments/7/runs/e37265739b1443ef83ec07809b791c14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


In [9]:
paragraphs_run2 = [
    "Elon Musk, CEO of Tesla and SpaceX, announced new projects in Austin, Texas during a global technology conference attended by major industry leaders.",
    "Microsoft and Google are competing heavily in artificial intelligence research across the United States, Europe, and Asia.",
    "Former US President Barack Obama delivered a keynote speech at a climate summit in Paris organized by the United Nations and European Union.",
    "Apple, headquartered in Cupertino, and Amazon, based in Seattle, dominate global consumer technology and cloud computing markets.",
    "The World Health Organization and World Bank held a joint meeting in Geneva to discuss global pandemic response strategies and funding.",
    "Tesla opened a new Gigafactory in Berlin, Germany while expanding production facilities in China and the United States.",
    "Mark Zuckerberg of Meta announced major policy changes in California during a press event attended by journalists worldwide.",
    "The European Union and United Nations collaborated on economic reforms in Brussels involving multiple member countries.",
    "Google DeepMind researchers in London published a breakthrough in machine learning models used for healthcare and robotics.",
    "Jeff Bezos, founder of Amazon, discussed space exploration plans with Blue Origin engineers in Washington during a private briefing."
]

entities_2 = []

with mlflow.start_run(run_name="ner_run_2_complex"):
    for p in paragraphs_run2:
        entities_2.extend(ner(p))

    labels = [e["entity_group"] for e in entities_2]
    counts = Counter(labels)

    mlflow.log_param("model_name", "dslim/bert-base-NER")
    mlflow.log_param("num_paragraphs", len(paragraphs_run2))
    mlflow.log_param("dataset_type", "complex_real_world_text")

    mlflow.log_metric("total_entities", len(entities_2))
    mlflow.log_metric("PER", counts.get("PER", 0))
    mlflow.log_metric("ORG", counts.get("ORG", 0))
    mlflow.log_metric("LOC", counts.get("LOC", 0))

    df = pd.DataFrame(entities_2)
    df.to_csv("ner_run2.csv", index=False)

    mlflow.log_artifact("ner_run2.csv")

🏃 View run ner_run_2_complex at: http://127.0.0.1:5000/#/experiments/7/runs/3df645f486a248f19de27f3ca312654a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


`Observations:`

`Run 1 — Easy Text`

Setup:

Dataset: simple, structured paragraphs
Paragraphs: 10
Model: dslim/bert-base-NER

Results:

Total entities: 26
PERSON (PER): 2
ORGANIZATION (ORG): 12
LOCATION (LOC): 9

Observations (Run 1):

The model performs well on simple and structured text, extracting entities with high confidence scores. Most entities are correctly identified, particularly organizations and locations. Some minor misclassifications are observed, such as "Elon Musk" being labeled as an organization instead of a person. Additionally, certain terms like "iPhone" and "MacBook" are classified as miscellaneous entities, which is expected since they do not belong to standard categories like person, organization, or location. Overall, the model shows strong performance on clean input with minimal ambiguity.

`Run 2 — Complex Real-World Text`

Setup:

Dataset: longer, more complex paragraphs
Paragraphs: 10
Same model used

Results:

Total entities: 41
PERSON (PER): 5
ORGANIZATION (ORG): 18
LOCATION (LOC): 18

Observations (Run 2):

When applied to more complex and realistic paragraphs, the model extracts a higher number of entities due to increased text richness and multiple entity mentions per paragraph. However, performance becomes less consistent. Some entities are split incorrectly (e.g., "Elon Musk" split into sub-tokens), and partial or incorrect labels appear (e.g., "Gigafa" instead of "Gigafactory"). Additionally, the model occasionally produces lower-confidence predictions and misclassifications due to overlapping context and longer sentence structures.

`Comparison`

Comparing both runs, the model performs more accurately on simple text, while complex real-world paragraphs lead to increased entity extraction but also more errors and inconsistencies. The rise in total entities (26 -> 41) reflects higher information density, but also introduces challenges such as token fragmentation and ambiguous labeling. This demonstrates that while pretrained NER models generalize well, their performance degrades with increased linguistic complexity and contextual ambiguity.

`Task 3: Text Generation`

In [3]:
import mlflow
import pandas as pd
from transformers import pipeline

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Text Generation")

generator = pipeline("text-generation", model="gpt2")

prompts = [
    "Artificial intelligence will",
    "The future of education is",
    "In a distant galaxy,",
    "Climate change is",
    "The most important invention in history is"
]

2026/04/20 01:55:56 INFO mlflow.tracking.fluent: Experiment with name 'Text Generation' does not exist. Creating a new experiment.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
results_run1 = []

with mlflow.start_run(run_name="gen_run_1_low_temp"):

    temperature = 0.7
    max_length = 50

    mlflow.log_param("model", "gpt2")
    mlflow.log_param("temperature", temperature)
    mlflow.log_param("max_length", max_length)

    for prompt in prompts:
        output = generator(prompt, max_length=max_length, temperature=temperature, num_return_sequences=1)
        generated_text = output[0]["generated_text"]

        results_run1.append({
            "prompt": prompt,
            "generated_text": generated_text
        })

    df1 = pd.DataFrame(results_run1)
    df1.to_csv("gen_run1.csv", index=False)
    mlflow.log_artifact("gen_run1.csv")

    # Observations
    obs1 = """Run 1 uses a lower temperature (0.7), resulting in more coherent and focused text generation.
The outputs are generally consistent but less creative and more repetitive."""
    
    with open("obs_run1.txt", "w") as f:
        f.write(obs1)

    mlflow.log_artifact("obs_run1.txt")

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Bo

🏃 View run gen_run_1_low_temp at: http://127.0.0.1:5000/#/experiments/8/runs/8d959fd0c5394fb882a316b0e080bbea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/8


In [5]:
results_run2 = []

with mlflow.start_run(run_name="gen_run_2_high_temp"):

    temperature = 1.2
    max_length = 80

    mlflow.log_param("model", "gpt2")
    mlflow.log_param("temperature", temperature)
    mlflow.log_param("max_length", max_length)

    for prompt in prompts:
        output = generator(prompt, max_length=max_length, temperature=temperature, num_return_sequences=1)
        generated_text = output[0]["generated_text"]

        results_run2.append({
            "prompt": prompt,
            "generated_text": generated_text
        })

    df2 = pd.DataFrame(results_run2)
    df2.to_csv("gen_run2.csv", index=False)
    mlflow.log_artifact("gen_run2.csv")

    # Observations
    obs2 = """Run 2 uses a higher temperature (1.2), leading to more diverse and creative outputs.
However, the generated text may be less coherent and sometimes grammatically inconsistent."""
    
    with open("obs_run2.txt", "w") as f:
        f.write(obs2)

    mlflow.log_artifact("obs_run2.txt")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_

🏃 View run gen_run_2_high_temp at: http://127.0.0.1:5000/#/experiments/8/runs/16ffe3df33904fa8931c32c29ec9c4ba
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/8


`Observations:`

`Run 1 — Low Temperature (0.7, max_length=50)`

With a lower temperature (0.7), the generated text is generally more coherent and structured. The outputs tend to follow logical sentence patterns and maintain topic consistency across the generated sequence. However, the model often becomes repetitive, as seen in examples like repeated mentions of "quantum computers" and "the automobile". Additionally, some outputs appear overly deterministic and less diverse, with similar phrasing patterns across different prompts. Overall, the model prioritizes fluency and coherence over creativity in this setting.

`Run 2 — High Temperature (1.2, max_length=80)`

Increasing the temperature to 1.2 results in more diverse and creative text generation. The outputs contain a wider range of ideas and less predictable phrasing, often introducing unexpected concepts and narrative directions. However, this increased randomness reduces coherence, leading to occasional inconsistencies, abrupt topic shifts, and less logically connected sentences. Some outputs also become overly verbose due to the increased maximum length, further amplifying variability. Overall, the model produces more imaginative but less controlled text under higher temperature settings.

`Comparison`

Comparing both runs, temperature significantly influences the trade-off between coherence and creativity in text generation. Lower temperature values produce more stable and grammatically consistent outputs but limit diversity, while higher temperature values enhance creativity at the cost of coherence and structure. The increase in max_length in Run 2 further contributes to longer and more varied outputs, but also introduces additional noise and inconsistency. This demonstrates the importance of parameter tuning in controlling the behavior of generative language models.